# Compute: CELLxGENE Large-scale Scalability Benchmark

Runs CrossCell conversion benchmarks on 11 CELLxGENE datasets (10k-1.2M cells).
Designed for a 128 GB cloud server.

**Operations**: `inspect`, `h5ad_to_rds`, `rds_to_h5ad` (roundtrip)

**Output**: `cellxgene_scalability.json`

Run this first, then `fig4_scalability.ipynb` for plotting.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import json, os, re, subprocess, time
from pathlib import Path
from datetime import datetime

DATA_DIR = Path('/benchmark/data/generated')
RESULTS_DIR = Path('/benchmark/results')
TMP_DIR = Path('/tmp/scalability_work')

for d in [RESULTS_DIR, TMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RESULTS_FILE = RESULTS_DIR / 'cellxgene_scalability.json'

print('\u2705 Environment setup complete')


✅ Environment setup complete


## 1. Helper Functions

In [2]:
def run_timed(cmd, timeout=7200):
    """Run command with /usr/bin/time to capture peak memory."""
    time_cmd = ['/usr/bin/time', '-v'] + (cmd if isinstance(cmd, list) else cmd.split())
    start = time.time()
    try:
        r = subprocess.run(time_cmd, capture_output=True, text=True, timeout=timeout)
        elapsed = time.time() - start
    except subprocess.TimeoutExpired:
        return False, timeout, 0, 'TIMEOUT'
    peak = 0
    m = re.search(r'Maximum resident set size \(kbytes\): (\d+)', r.stderr)
    if m:
        peak = int(m.group(1)) / 1024.0
    return r.returncode == 0, elapsed, peak, r.stderr

def file_size_mb(path):
    """Get file size in MB."""
    p = Path(path)
    return round(p.stat().st_size / (1024*1024), 1) if p.exists() else None

print('\u2705 Helper functions defined')


✅ Helper functions defined


## 2. Dataset Definitions

In [3]:
DATASETS = [
    {'name': 'skin_bcc_10k',        'cells': 9841,    'genes': 26886},
    {'name': 'tabula_liver_22k',     'cells': 22214,   'genes': 60606},
    {'name': 'kidney_atacseq_37k',   'cells': 37747,   'genes': 19276},
    {'name': 'brain_multiome_102k',  'cells': 101924,  'genes': 35451},
    {'name': 'pancreas_122k',        'cells': 121916,  'genes': 32356},
    {'name': 'brain_dlpfc_172k',     'cells': 172120,  'genes': 37490},
    {'name': 'gut_428k',             'cells': 428469,  'genes': 32383},
    {'name': 'heart_486k',           'cells': 486134,  'genes': 32383},
    {'name': 'hlca_core_585k',       'cells': 584944,  'genes': 27402},
    {'name': 'combat_pbmc_836k',     'cells': 836148,  'genes': 36306},
    {'name': 'eqtl_autoimmune_1.2M', 'cells': 1248980, 'genes': 35528},
]

# Check which files exist
for ds in DATASETS:
    h5ad = DATA_DIR / f"cellxgene_{ds['name']}.h5ad"
    icon = '\u2705' if h5ad.exists() else '\u274c'
    ds['h5ad_path'] = str(h5ad)
    print(f"  {icon} {ds['name']:30s} {h5ad.name}")

print(f'\nTotal: {len(DATASETS)} datasets')


  ✅ skin_bcc_10k                   cellxgene_skin_bcc_10k.h5ad
  ✅ tabula_liver_22k               cellxgene_tabula_liver_22k.h5ad
  ✅ kidney_atacseq_37k             cellxgene_kidney_atacseq_37k.h5ad
  ✅ brain_multiome_102k            cellxgene_brain_multiome_102k.h5ad
  ✅ pancreas_122k                  cellxgene_pancreas_122k.h5ad
  ✅ brain_dlpfc_172k               cellxgene_brain_dlpfc_172k.h5ad
  ✅ gut_428k                       cellxgene_gut_428k.h5ad
  ✅ heart_486k                     cellxgene_heart_486k.h5ad
  ✅ hlca_core_585k                 cellxgene_hlca_core_585k.h5ad
  ✅ combat_pbmc_836k               cellxgene_combat_pbmc_836k.h5ad
  ✅ eqtl_autoimmune_1.2M           cellxgene_eqtl_autoimmune_1.2M.h5ad

Total: 11 datasets


## 3. Run Benchmarks

In [4]:
results = []

for ds in DATASETS:
    name = ds['name']
    h5ad_path = ds['h5ad_path']
    rds_path = str(TMP_DIR / f'{name}.rds')
    rt_path = str(TMP_DIR / f'{name}_roundtrip.h5ad')

    print(f'\n{"="*60}')
    print(f'Dataset: {name} ({ds["cells"]:,} cells)')
    print(f'{"="*60}')

    rec = {'name': name, 'cells': ds['cells'], 'genes': ds['genes'],
           'file_mb': file_size_mb(h5ad_path), 'error': ''}

    if not Path(h5ad_path).exists():
        print(f'  \u274c File not found: {h5ad_path}')
        rec['error'] = 'file_not_found'
        for op in ['inspect','h5ad_to_rds','rds_to_h5ad']:
            rec[op] = {'time_s':0,'peak_mem_mb':0,'status':'skipped'}
        results.append(rec)
        continue

    # ── inspect ──
    print(f'  [1/3] inspect ...')
    ok, t, mem, stderr = run_timed(['crosscell','inspect','-i',h5ad_path])
    rec['inspect'] = {'time_s':round(t,2),'peak_mem_mb':round(mem,1),
                       'status':'success' if ok else 'failed'}
    icon = '\u2705' if ok else '\u274c'
    print(f'  {icon} inspect: {t:.1f}s, {mem:.0f} MB')

    # ── h5ad_to_rds ──
    print(f'  [2/3] h5ad_to_rds ...')
    ok, t, mem, stderr = run_timed(['crosscell','convert','-i',h5ad_path,'-o',rds_path,'-f','seurat'])
    rec['h5ad_to_rds'] = {'time_s':round(t,2),'peak_mem_mb':round(mem,1),
                           'rds_mb':file_size_mb(rds_path),'status':'success' if ok else 'failed'}
    icon = '\u2705' if ok else '\u274c'
    print(f'  {icon} h5ad_to_rds: {t:.1f}s, {mem:.0f} MB')
    if not ok:
        rec['error'] = stderr[-200:] if stderr else 'unknown'

    # ── rds_to_h5ad (roundtrip) ──
    if ok:
        print(f'  [3/3] rds_to_h5ad (roundtrip) ...')
        ok2, t2, mem2, stderr2 = run_timed(['crosscell','convert','-i',rds_path,'-o',rt_path,'-f','anndata'])
        rec['rds_to_h5ad'] = {'time_s':round(t2,2),'peak_mem_mb':round(mem2,1),
                               'h5ad_mb':file_size_mb(rt_path),'status':'success' if ok2 else 'failed'}
        icon = '\u2705' if ok2 else '\u274c'
        print(f'  {icon} rds_to_h5ad: {t2:.1f}s, {mem2:.0f} MB')
        if not ok2:
            rec['error'] = stderr2[-200:] if stderr2 else 'unknown'
    else:
        rec['rds_to_h5ad'] = {'time_s':0,'peak_mem_mb':0,'status':'skipped'}

    results.append(rec)

print(f'\n\u2705 All benchmarks complete ({len(results)} datasets)')



Dataset: skin_bcc_10k (9,841 cells)
  [1/3] inspect ...
  ✅ inspect: 1.3s, 445 MB
  [2/3] h5ad_to_rds ...
  ✅ h5ad_to_rds: 49.0s, 1085 MB
  [3/3] rds_to_h5ad (roundtrip) ...
  ✅ rds_to_h5ad: 3.6s, 1081 MB

Dataset: tabula_liver_22k (22,214 cells)
  [1/3] inspect ...
  ✅ inspect: 11.7s, 3436 MB
  [2/3] h5ad_to_rds ...
  ✅ h5ad_to_rds: 566.9s, 7062 MB
  [3/3] rds_to_h5ad (roundtrip) ...
  ✅ rds_to_h5ad: 38.8s, 8388 MB

Dataset: kidney_atacseq_37k (37,747 cells)
  [1/3] inspect ...
  ✅ inspect: 11.6s, 5174 MB
  [2/3] h5ad_to_rds ...
  ✅ h5ad_to_rds: 611.2s, 12854 MB
  [3/3] rds_to_h5ad (roundtrip) ...
  ✅ rds_to_h5ad: 44.5s, 12854 MB

Dataset: brain_multiome_102k (101,924 cells)
  [1/3] inspect ...
  ✅ inspect: 14.6s, 5793 MB
  [2/3] h5ad_to_rds ...
  ✅ h5ad_to_rds: 676.1s, 14220 MB
  [3/3] rds_to_h5ad (roundtrip) ...
  ✅ rds_to_h5ad: 54.2s, 14336 MB

Dataset: pancreas_122k (121,916 cells)
  [1/3] inspect ...
  ✅ inspect: 9.7s, 3751 MB
  [2/3] h5ad_to_rds ...
  ✅ h5ad_to_rds: 423.2s, 921

## 4. Save Results

In [5]:
output = {
    'metadata': {
        'timestamp': datetime.now().isoformat(),
        'machine': '128GB cloud server',
        'crosscell_version': '0.1.x',
    },
    'results': results,
}

with open(RESULTS_FILE, 'w') as f:
    json.dump(output, f, indent=2)

print(f'\u2705 Results saved to {RESULTS_FILE}')
print(f'Total datasets: {len(results)}')

# Summary
for r in results:
    h2r = r.get('h5ad_to_rds',{})
    r2h = r.get('rds_to_h5ad',{})
    print(f"  {r['name']:30s} inspect={r.get('inspect',{}).get('status','?'):8s} "
          f"h2r={h2r.get('status','?'):8s} r2h={r2h.get('status','?'):8s}")


✅ Results saved to /benchmark/results/cellxgene_scalability.json
Total datasets: 11
  skin_bcc_10k                   inspect=success  h2r=success  r2h=success 
  tabula_liver_22k               inspect=success  h2r=success  r2h=success 
  kidney_atacseq_37k             inspect=success  h2r=success  r2h=success 
  brain_multiome_102k            inspect=success  h2r=success  r2h=success 
  pancreas_122k                  inspect=success  h2r=success  r2h=success 
  brain_dlpfc_172k               inspect=success  h2r=success  r2h=success 
  gut_428k                       inspect=success  h2r=success  r2h=success 
  heart_486k                     inspect=success  h2r=success  r2h=success 
  hlca_core_585k                 inspect=success  h2r=success  r2h=success 
  combat_pbmc_836k               inspect=success  h2r=success  r2h=success 
  eqtl_autoimmune_1.2M           inspect=success  h2r=success  r2h=success 
